In [ ]:
# -*- coding: utf-8 -*-
"""
AUTOENCODER COM CURVA COMO ENTRADA — SEM TEMPERATURA COMO INPUT — SEM PCA

Objetivo:
- Entrada da rede: curva medida.
- Saída da rede: curva compensada para REF_TEMP.
- A temperatura NÃO entra como input.
- A falha NÃO entra como input.
- A falha entra apenas como alvo auxiliar de treino.
- A temperatura entra apenas como alvo auxiliar de treino.
- Não usa PCA.
- Não usa tipo de dano como entrada no teste.
"""

# ============================================================
# 1) IMPORTS
# ============================================================

import os
import re
import time
import copy
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    f1_score
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# 2) PARÂMETROS
# ============================================================

ARQ_BASE = "base-completo--.pkl"

REF_TEMP = 30

FREQ_MIN_KHZ = 40
FREQ_MAX_KHZ = 50

OUTPUT_DIR = f"AE_CurveInput_REF{REF_TEMP}C_{FREQ_MIN_KHZ}-{FREQ_MAX_KHZ}kHz"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ---------------- PARK PARA COMPARAÇÃO ----------------

PARK_MAX_SHIFT_FRAC = 0.10
PARK_NSTEPS = 101
PARK_SMOOTH_WIN = 1


# ---------------- AUTOENCODER COM CURVA COMO ENTRADA ----------------

AE_EPOCHS = 1000
AE_BATCH_SIZE = 8
AE_LR = 8e-4
AE_PATIENCE = 150

LATENT_DIM = 128

ALPHA_COMP_AE = 0.90

RESIDUAL_PERCENTILE = 99.0
RESIDUAL_SCALE_MIN = 0.30
RESIDUAL_SCALE_MAX = 3.00

LAMBDA_CORR = 0.10
LAMBDA_DERIV = 0.20
LAMBDA_SMOOTH_DELTA = 0.005
LAMBDA_DMG = 0.20
LAMBDA_TEMP = 0.05

USE_TEMP_AUX_TARGET = True

NUM_WORKERS = 0

PLOT_EXEMPLOS = ((0, 48), (1, 55), (2, 55))
HIST_BINS = 18

np.random.seed(42)
torch.manual_seed(42)


# ============================================================
# 3) FUNÇÕES GERAIS
# ============================================================

def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_freq_columns(df, fmin_khz, fmax_khz):
    cols = []
    freqs = []

    for c in df.columns:
        f = extract_freq_hz(c)

        if f is not None:
            f_khz = f / 1e3

            if fmin_khz <= f_khz <= fmax_khz:
                cols.append(c)
                freqs.append(f)

    order = np.argsort(freqs)

    fcols = [cols[i] for i in order]
    fhz = np.array(freqs, dtype=float)[order]

    return fcols, fhz


def moving_average(arr, win):
    if win <= 1 or win % 2 == 0:
        return arr.copy()

    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode="edge")
    kernel = np.ones(win) / win
    smooth = np.convolve(arr_pad, kernel, mode="valid")

    return smooth[:len(arr)]


def get_reference_curves_by_damage(df, fcols, ref_temp):
    ref_by_damage = {}

    damages = sorted(df["falha"].unique())

    for d in damages:
        pool = df.loc[
            (df["falha"] == d) &
            (np.isclose(df["temperatura_c"], ref_temp)),
            fcols
        ].to_numpy(float)

        if len(pool) == 0:
            raise ValueError(
                f"Nenhuma curva encontrada para falha={d} em REF_TEMP={ref_temp} °C."
            )

        ref_by_damage[d] = np.median(pool, axis=0)

    return ref_by_damage


def construir_target_por_dano(df, fcols, ref_by_damage):
    y_target = []

    for _, row in df.iterrows():
        d = row["falha"]
        y_target.append(ref_by_damage[d])

    y_target = np.asarray(y_target, dtype=float)

    return y_target


def aplicar_estilo_artigo():
    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": 18,
        "axes.labelsize": 20,
        "axes.titlesize": 20,
        "xtick.labelsize": 17,
        "ytick.labelsize": 17,
        "legend.fontsize": 15,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    })


# ============================================================
# 4) MÉTRICAS
# ============================================================

def rmsd(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)

    return float(np.sqrt(np.mean((y - ref) ** 2)))


def ccdm(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)

    y0 = y - np.mean(y)
    r0 = ref - np.mean(ref)

    num = float(np.sum(y0 * r0))
    den = float(np.sqrt(np.sum(y0 ** 2) * np.sum(r0 ** 2)) + 1e-18)

    corr = num / den

    return float(1 - corr)


def calcular_metricas(df_curvas, fcols, y_ref_healthy, metodo):
    X = df_curvas[fcols].to_numpy(float)

    df_out = df_curvas.copy()

    df_out["RMSD"] = [rmsd(x, y_ref_healthy) for x in X]
    df_out["CCDM"] = [ccdm(x, y_ref_healthy) for x in X]
    df_out["Metodo"] = metodo

    return df_out


def corr_loss_batch(y_pred, y_true):
    yp = y_pred.squeeze(1)
    yt = y_true.squeeze(1)

    yp0 = yp - yp.mean(dim=1, keepdim=True)
    yt0 = yt - yt.mean(dim=1, keepdim=True)

    num = torch.sum(yp0 * yt0, dim=1)
    den = torch.sqrt(
        torch.sum(yp0 ** 2, dim=1) *
        torch.sum(yt0 ** 2, dim=1) +
        1e-12
    )

    corr = num / den

    return torch.mean(1.0 - corr)


def derivative_loss(y_pred, y_true):
    dy_pred = y_pred[:, :, 1:] - y_pred[:, :, :-1]
    dy_true = y_true[:, :, 1:] - y_true[:, :, :-1]

    return F.smooth_l1_loss(dy_pred, dy_true)


def smooth_delta_loss(delta):
    d_delta = delta[:, :, 1:] - delta[:, :, :-1]

    return torch.mean(d_delta ** 2)


# ============================================================
# 5) PARK
# ============================================================

def shift_interp(x, fHz, tau):
    f_shift = fHz + tau

    return np.interp(
        fHz,
        f_shift,
        x,
        left=x[0],
        right=x[-1]
    )


def park_single(x, ref, fHz):
    df_band = fHz[-1] - fHz[0]
    tau_max = PARK_MAX_SHIFT_FRAC * df_band

    best_err = np.inf
    best_tau = 0.0
    best_dS = 0.0

    taus = np.linspace(-tau_max, tau_max, PARK_NSTEPS)

    for tau in taus:
        x_shift = shift_interp(x, fHz, tau)

        dS = np.mean(ref - x_shift)

        y_try = x_shift + dS

        err = np.mean((ref - y_try) ** 2)

        if err < best_err:
            best_err = err
            best_tau = tau
            best_dS = dS

    y_comp = shift_interp(x, fHz, best_tau) + best_dS
    y_comp = moving_average(y_comp, PARK_SMOOTH_WIN)

    return y_comp


def compensar_park(df, fcols, fHz, y_ref_healthy):
    print("\n====================================================")
    print("APLICANDO PARK")
    print("====================================================")

    X_all = df[fcols].to_numpy(float)
    Y_comp = np.zeros_like(X_all)

    for i in range(len(X_all)):
        Y_comp[i] = park_single(
            x=X_all[i],
            ref=y_ref_healthy,
            fHz=fHz
        )

        if (i + 1) % 20 == 0 or (i + 1) == len(X_all):
            print(f"Park: {i+1}/{len(X_all)} curvas compensadas")

    df_comp = df.copy()
    df_comp[fcols] = Y_comp

    return df_comp


# ============================================================
# 6) MODELO AUTOENCODER COM CURVA COMO ENTRADA
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv1d(cin, cout, kernel_size=5, stride=stride, padding=2),
            nn.BatchNorm1d(cout),
            nn.GELU(),
            nn.Conv1d(cout, cout, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(cout),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)


class CurveInputAutoencoder(nn.Module):
    def __init__(
        self,
        n_points,
        n_classes,
        latent_dim=128,
        residual_scale=1.0
    ):
        super().__init__()

        self.n_points = int(n_points)
        self.n_classes = int(n_classes)
        self.latent_dim = int(latent_dim)
        self.residual_scale = float(residual_scale)

        self.down_factor = 32
        self.n_pad = int(np.ceil(self.n_points / self.down_factor) * self.down_factor)
        self.down_len = self.n_pad // self.down_factor

        self.enc1 = ConvBlock(1, 16, stride=2)
        self.enc2 = ConvBlock(16, 32, stride=2)
        self.enc3 = ConvBlock(32, 64, stride=2)
        self.enc4 = ConvBlock(64, 96, stride=2)
        self.enc5 = ConvBlock(96, 128, stride=2)

        enc_flat = 128 * self.down_len

        self.fc_enc = nn.Sequential(
            nn.Linear(enc_flat, 512),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(512, latent_dim),
            nn.GELU()
        )

        self.fc_dec = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.GELU(),
            nn.Linear(512, enc_flat),
            nn.GELU()
        )

        self.dec5 = ConvBlock(128, 96, stride=1)
        self.dec4 = ConvBlock(96, 64, stride=1)
        self.dec3 = ConvBlock(64, 32, stride=1)
        self.dec2 = ConvBlock(32, 16, stride=1)
        self.dec1 = ConvBlock(16, 8, stride=1)

        self.out_delta = nn.Conv1d(8, 1, kernel_size=5, padding=2)

        self.damage_head = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(64, n_classes)
        )

        self.temp_head = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(64, 1)
        )

    def pad_input(self, x):
        L = x.shape[-1]

        if L == self.n_pad:
            return x

        pad_right = self.n_pad - L

        return F.pad(x, (0, pad_right), mode="replicate")

    def crop_output(self, x):
        return x[:, :, :self.n_points]

    def forward(self, x):
        x_in = x

        xp = self.pad_input(x)

        z = self.enc1(xp)
        z = self.enc2(z)
        z = self.enc3(z)
        z = self.enc4(z)
        z = self.enc5(z)

        z_flat = z.reshape(z.shape[0], -1)

        latent = self.fc_enc(z_flat)

        damage_logits = self.damage_head(latent)
        temp_pred = self.temp_head(latent)

        dec = self.fc_dec(latent)
        dec = dec.reshape(x.shape[0], 128, self.down_len)

        dec = F.interpolate(dec, scale_factor=2, mode="linear", align_corners=False)
        dec = self.dec5(dec)

        dec = F.interpolate(dec, scale_factor=2, mode="linear", align_corners=False)
        dec = self.dec4(dec)

        dec = F.interpolate(dec, scale_factor=2, mode="linear", align_corners=False)
        dec = self.dec3(dec)

        dec = F.interpolate(dec, scale_factor=2, mode="linear", align_corners=False)
        dec = self.dec2(dec)

        dec = F.interpolate(dec, scale_factor=2, mode="linear", align_corners=False)
        dec = self.dec1(dec)

        raw_delta = self.out_delta(dec)
        raw_delta = self.crop_output(raw_delta)

        delta = self.residual_scale * torch.tanh(raw_delta)

        y_comp = x_in + ALPHA_COMP_AE * delta

        return y_comp, delta, damage_logits, temp_pred


# ============================================================
# 7) TREINO DO AUTOENCODER
# ============================================================

def preparar_dados_autoencoder(df, fcols, ref_by_damage):
    X = df[fcols].to_numpy(float)

    Y = construir_target_por_dano(
        df=df,
        fcols=fcols,
        ref_by_damage=ref_by_damage
    )

    falhas_orig = df["falha"].to_numpy()

    classes = sorted(np.unique(falhas_orig))
    class_to_idx = {c: i for i, c in enumerate(classes)}
    idx_to_class = {i: c for c, i in class_to_idx.items()}

    y_class = np.array([class_to_idx[c] for c in falhas_orig], dtype=int)

    T = df["temperatura_c"].to_numpy(float)

    T_aux = ((T - REF_TEMP) / 100.0).reshape(-1, 1)

    return X, Y, y_class, T_aux, class_to_idx, idx_to_class


def train_curve_autoencoder(df, fcols, ref_by_damage):
    print("\n====================================================")
    print("TREINANDO AUTOENCODER COM CURVA COMO ENTRADA")
    print("====================================================")
    print("Input da rede: curva")
    print("Temperatura NÃO entra como input")
    print("Falha NÃO entra como input")
    print("Falha entra apenas como alvo auxiliar de treino")
    print("Temperatura entra apenas como alvo auxiliar de treino")

    X, Y, y_class, T_aux, class_to_idx, idx_to_class = preparar_dados_autoencoder(
        df=df,
        fcols=fcols,
        ref_by_damage=ref_by_damage
    )

    indices = np.arange(len(df))

    idx_train, idx_val = train_test_split(
        indices,
        test_size=0.20,
        random_state=42,
        stratify=y_class
    )

    scaler = StandardScaler()

    scaler.fit(X[idx_train])

    Xs = scaler.transform(X)
    Ys = scaler.transform(Y)

    residual_train = Ys[idx_train] - Xs[idx_train]
    residual_scale = float(np.percentile(np.abs(residual_train), RESIDUAL_PERCENTILE))
    residual_scale = float(np.clip(residual_scale, RESIDUAL_SCALE_MIN, RESIDUAL_SCALE_MAX))

    print(f"\nResidual scale usado no tanh: {residual_scale:.4f}")

    X_tensor = torch.tensor(Xs[:, None, :], dtype=torch.float32)
    Y_tensor = torch.tensor(Ys[:, None, :], dtype=torch.float32)
    C_tensor = torch.tensor(y_class, dtype=torch.long)
    T_tensor = torch.tensor(T_aux, dtype=torch.float32)

    train_dataset = TensorDataset(
        X_tensor[idx_train],
        Y_tensor[idx_train],
        C_tensor[idx_train],
        T_tensor[idx_train]
    )

    val_dataset = TensorDataset(
        X_tensor[idx_val],
        Y_tensor[idx_val],
        C_tensor[idx_val],
        T_tensor[idx_val]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=AE_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=AE_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Dispositivo usado no Autoencoder: {device}")

    model = CurveInputAutoencoder(
        n_points=X.shape[1],
        n_classes=len(class_to_idx),
        latent_dim=LATENT_DIM,
        residual_scale=residual_scale
    ).to(device)

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=AE_LR,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="min",
        factor=0.5,
        patience=40
    )

    ce_loss = nn.CrossEntropyLoss()
    mse_loss = nn.MSELoss()
    huber_loss = nn.SmoothL1Loss()

    best_val = np.inf
    best_state = copy.deepcopy(model.state_dict())
    epochs_sem_melhora = 0

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_curve": [],
        "val_damage_acc": []
    }

    for ep in range(1, AE_EPOCHS + 1):
        model.train()

        train_losses = []

        for xb, yb, cb, tb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            cb = cb.to(device)
            tb = tb.to(device)

            opt.zero_grad()

            pred, delta, logits, tpred = model(xb)

            loss_curve = huber_loss(pred, yb)
            loss_corr = corr_loss_batch(pred, yb)
            loss_deriv = derivative_loss(pred, yb)
            loss_smooth = smooth_delta_loss(delta)
            loss_dmg = ce_loss(logits, cb)

            if USE_TEMP_AUX_TARGET:
                loss_temp = mse_loss(tpred, tb)
            else:
                loss_temp = torch.tensor(0.0, device=device)

            loss = (
                loss_curve
                + LAMBDA_CORR * loss_corr
                + LAMBDA_DERIV * loss_deriv
                + LAMBDA_SMOOTH_DELTA * loss_smooth
                + LAMBDA_DMG * loss_dmg
                + LAMBDA_TEMP * loss_temp
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            opt.step()

            train_losses.append(loss.item())

        train_loss_mean = float(np.mean(train_losses))

        model.eval()

        val_losses = []
        val_curve_losses = []
        all_pred_class = []
        all_true_class = []

        with torch.no_grad():
            for xb, yb, cb, tb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                cb = cb.to(device)
                tb = tb.to(device)

                pred, delta, logits, tpred = model(xb)

                loss_curve = huber_loss(pred, yb)
                loss_corr = corr_loss_batch(pred, yb)
                loss_deriv = derivative_loss(pred, yb)
                loss_smooth = smooth_delta_loss(delta)
                loss_dmg = ce_loss(logits, cb)

                if USE_TEMP_AUX_TARGET:
                    loss_temp = mse_loss(tpred, tb)
                else:
                    loss_temp = torch.tensor(0.0, device=device)

                loss = (
                    loss_curve
                    + LAMBDA_CORR * loss_corr
                    + LAMBDA_DERIV * loss_deriv
                    + LAMBDA_SMOOTH_DELTA * loss_smooth
                    + LAMBDA_DMG * loss_dmg
                    + LAMBDA_TEMP * loss_temp
                )

                val_losses.append(loss.item())
                val_curve_losses.append(loss_curve.item())

                pred_class = torch.argmax(logits, dim=1)

                all_pred_class.extend(pred_class.cpu().numpy().tolist())
                all_true_class.extend(cb.cpu().numpy().tolist())

        val_loss_mean = float(np.mean(val_losses))
        val_curve_mean = float(np.mean(val_curve_losses))
        val_acc = accuracy_score(all_true_class, all_pred_class)

        scheduler.step(val_loss_mean)

        history["epoch"].append(ep)
        history["train_loss"].append(train_loss_mean)
        history["val_loss"].append(val_loss_mean)
        history["val_curve"].append(val_curve_mean)
        history["val_damage_acc"].append(val_acc)

        if val_loss_mean < best_val - 1e-7:
            best_val = val_loss_mean
            best_state = copy.deepcopy(model.state_dict())
            epochs_sem_melhora = 0
        else:
            epochs_sem_melhora += 1

        if ep == 1 or ep % 25 == 0:
            print(
                f"Epoch {ep:4d}/{AE_EPOCHS} | "
                f"train={train_loss_mean:.6f} | "
                f"val={val_loss_mean:.6f} | "
                f"curve={val_curve_mean:.6f} | "
                f"dmg_acc={val_acc:.4f}"
            )

        if epochs_sem_melhora >= AE_PATIENCE:
            print(
                f"Early stopping na epoch {ep}. "
                f"Melhor val_loss = {best_val:.6f}"
            )
            break

    model.load_state_dict(best_state)

    history = pd.DataFrame(history)

    extra = {
        "model": model,
        "scaler": scaler,
        "device": device,
        "history": history,
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "idx_train": idx_train,
        "idx_val": idx_val,
        "residual_scale": residual_scale
    }

    return extra


def aplicar_autoencoder_em_todas_curvas(df, fcols, ae_extra):
    print("\n====================================================")
    print("APLICANDO AUTOENCODER EM TODAS AS CURVAS")
    print("====================================================")

    model = ae_extra["model"]
    scaler = ae_extra["scaler"]
    device = ae_extra["device"]
    idx_to_class = ae_extra["idx_to_class"]

    X = df[fcols].to_numpy(float)

    Xs = scaler.transform(X)

    X_tensor = torch.tensor(Xs[:, None, :], dtype=torch.float32)

    model.eval()

    preds_scaled = []
    logits_all = []
    temp_pred_all = []

    batch_size = AE_BATCH_SIZE

    with torch.no_grad():
        for i in range(0, len(X_tensor), batch_size):
            xb = X_tensor[i:i + batch_size].to(device)

            pred, delta, logits, tpred = model(xb)

            preds_scaled.append(pred.cpu().numpy()[:, 0, :])
            logits_all.append(logits.cpu().numpy())
            temp_pred_all.append(tpred.cpu().numpy())

    preds_scaled = np.vstack(preds_scaled)
    logits_all = np.vstack(logits_all)
    temp_pred_all = np.vstack(temp_pred_all)

    Y_comp = scaler.inverse_transform(preds_scaled)

    pred_class_idx = np.argmax(logits_all, axis=1)
    pred_damage = np.array([idx_to_class[i] for i in pred_class_idx])

    pred_temp_c = temp_pred_all[:, 0] * 100.0 + REF_TEMP

    df_comp = df.copy()
    df_comp[fcols] = Y_comp
    df_comp["falha_pred_aux"] = pred_damage
    df_comp["temperatura_pred_aux"] = pred_temp_c

    return df_comp


def compensar_autoencoder_curva_input(df, fcols, ref_by_damage):
    ae_extra = train_curve_autoencoder(
        df=df,
        fcols=fcols,
        ref_by_damage=ref_by_damage
    )

    df_comp = aplicar_autoencoder_em_todas_curvas(
        df=df,
        fcols=fcols,
        ae_extra=ae_extra
    )

    return df_comp, ae_extra


# ============================================================
# 8) RESUMOS
# ============================================================

def resumo_geral(df_long, metodos=("Park", "Autoencoder")):
    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    tabela = (
        df_use
        .groupby(["Metodo", "falha"])[["RMSD", "CCDM"]]
        .agg(["mean", "std", "min", "max"])
        .round(6)
    )

    return tabela


def resumo_por_temperatura(df_long, metodos=("Park", "Autoencoder")):
    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    tabela = (
        df_use
        .groupby(["Metodo", "temperatura_c", "falha"])[["RMSD", "CCDM"]]
        .mean()
        .reset_index()
        .sort_values(["Metodo", "temperatura_c", "falha"])
    )

    return tabela


def checar_monotonicidade(df_long, metodos=("Park", "Autoencoder")):
    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    registros = []

    for metodo in sorted(df_use["Metodo"].unique()):
        df_m = df_use[df_use["Metodo"] == metodo]

        temps = sorted(df_m["temperatura_c"].unique())

        for T in temps:
            df_t = df_m[np.isclose(df_m["temperatura_c"], T)]

            danos_presentes = set(df_t["falha"].unique())

            if not {0, 1, 2}.issubset(danos_presentes):
                continue

            for metrica in ["RMSD", "CCDM"]:
                medias = {}

                for d in [0, 1, 2]:
                    medias[d] = df_t.loc[
                        df_t["falha"] == d,
                        metrica
                    ].mean()

                ok = medias[0] < medias[1] < medias[2]

                registros.append({
                    "Metodo": metodo,
                    "Temperatura": T,
                    "Metrica": metrica,
                    "D0": medias[0],
                    "D1": medias[1],
                    "D2": medias[2],
                    "Monotonico_D0_D1_D2": ok
                })

    df_mono = pd.DataFrame(registros)

    if len(df_mono) == 0:
        resumo = pd.DataFrame()
        return df_mono, resumo

    resumo = (
        df_mono
        .groupby(["Metodo", "Metrica"])["Monotonico_D0_D1_D2"]
        .mean()
        .mul(100)
        .reset_index()
        .rename(columns={
            "Monotonico_D0_D1_D2": "Percentual_monotonico_%"
        })
    )

    return df_mono, resumo


# ============================================================
# 9) GRÁFICOS
# ============================================================

def plot_metricas_por_temperatura(
    df_long,
    dano=0,
    metodos=("Park", "Autoencoder"),
    salvar=True
):
    aplicar_estilo_artigo()

    df_use = df_long[
        (df_long["falha"] == dano) &
        (df_long["Metodo"].isin(metodos))
    ].copy()

    fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), dpi=300)

    metricas = ["RMSD", "CCDM"]

    for ax, metrica in zip(axes, metricas):

        for metodo in metodos:
            df_m = df_use[df_use["Metodo"] == metodo]

            g = (
                df_m
                .groupby("temperatura_c")[metrica]
                .mean()
                .reset_index()
                .sort_values("temperatura_c")
            )

            ax.plot(
                g["temperatura_c"],
                g[metrica],
                marker="o",
                linewidth=2,
                label=metodo
            )

        ax.set_xlabel("Temperatura (°C)")
        ax.set_ylabel(metrica)

        ax.grid(False)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_title(f"(a) RMSD — Dano {dano}")
    axes[1].set_title(f"(b) CCDM — Dano {dano}")

    axes[0].legend(
        frameon=True,
        facecolor="white",
        edgecolor="none"
    )

    plt.tight_layout()

    if salvar:
        caminho = os.path.join(
            OUTPUT_DIR,
            f"Metricas_por_temperatura_Dano{dano}.png"
        )

        plt.savefig(caminho, bbox_inches="tight", facecolor="white")

        print(f"Figura salva em: {caminho}")

    plt.show()


def plot_metricas_medias_por_dano(
    df_long,
    metodos=("Park", "Autoencoder"),
    salvar=True
):
    aplicar_estilo_artigo()

    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    danos = sorted(df_use["falha"].unique())

    fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), dpi=300)

    metricas = ["RMSD", "CCDM"]

    x = np.arange(len(danos))
    bar_w = 0.32

    offsets = np.linspace(
        -bar_w / 2,
        bar_w / 2,
        len(metodos)
    )

    for ax, metrica in zip(axes, metricas):

        for i, metodo in enumerate(metodos):
            vals = []
            stds = []

            for d in danos:
                mask = (
                    (df_use["Metodo"] == metodo) &
                    (df_use["falha"] == d)
                )

                vals.append(df_use.loc[mask, metrica].mean())
                stds.append(df_use.loc[mask, metrica].std())

            ax.bar(
                x + offsets[i],
                vals,
                width=bar_w,
                yerr=stds,
                capsize=4,
                edgecolor="black",
                linewidth=0.7,
                label=metodo
            )

        ax.set_xlabel("Classe de dano")
        ax.set_ylabel(metrica)

        ax.set_xticks(x)
        ax.set_xticklabels([f"Dano {d}" for d in danos])

        ax.grid(False)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_title("(a) RMSD médio por dano")
    axes[1].set_title("(b) CCDM médio por dano")

    axes[0].legend(
        frameon=True,
        facecolor="white",
        edgecolor="none"
    )

    plt.tight_layout()

    if salvar:
        caminho = os.path.join(
            OUTPUT_DIR,
            "Metricas_medias_por_dano.png"
        )

        plt.savefig(caminho, bbox_inches="tight", facecolor="white")

        print(f"Figura salva em: {caminho}")

    plt.show()


def plot_histogramas_metricas_grid(
    df_long,
    metodos=("Park", "Autoencoder"),
    bins=18,
    salvar=True
):
    aplicar_estilo_artigo()

    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    danos = sorted(df_use["falha"].unique())

    metricas = ["RMSD", "CCDM"]

    fig, axes = plt.subplots(
        len(danos),
        2,
        figsize=(17, 5.2 * len(danos)),
        dpi=300
    )

    if len(danos) == 1:
        axes = np.array([axes])

    for i, dano in enumerate(danos):

        df_d = df_use[df_use["falha"] == dano]

        for j, metrica in enumerate(metricas):

            ax = axes[i, j]

            for metodo in metodos:
                vals = df_d.loc[
                    df_d["Metodo"] == metodo,
                    metrica
                ].dropna().values

                if len(vals) == 0:
                    continue

                ax.hist(
                    vals,
                    bins=bins,
                    alpha=0.55,
                    edgecolor="black",
                    linewidth=0.7,
                    label=metodo
                )

                ax.axvline(
                    np.mean(vals),
                    linestyle="--",
                    linewidth=2
                )

            ax.set_xlabel(metrica)
            ax.set_ylabel("Frequência")
            ax.set_title(f"{metrica} — Dano {dano}")

            ax.grid(False)

            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            if i == 0 and j == 0:
                ax.legend(
                    frameon=True,
                    facecolor="white",
                    edgecolor="none"
                )

    plt.tight_layout()

    if salvar:
        caminho = os.path.join(
            OUTPUT_DIR,
            "Histogramas_Grid_Danos.png"
        )

        plt.savefig(caminho, bbox_inches="tight", facecolor="white")

        print(f"Figura salva em: {caminho}")

    plt.show()


def selecionar_indice_exemplo(df_base, falha, temperatura=None):
    df_aux = df_base[df_base["falha"] == falha].copy()

    if len(df_aux) == 0:
        raise ValueError(f"Nenhuma curva encontrada para falha = {falha}")

    if temperatura is None:
        return df_aux.index[0]

    df_temp = df_aux[np.isclose(df_aux["temperatura_c"], temperatura)]

    if len(df_temp) > 0:
        return df_temp.index[0]

    idx_mais_proximo = (
        (df_aux["temperatura_c"] - temperatura)
        .abs()
        .idxmin()
    )

    return idx_mais_proximo


def plot_curva_exemplo_individual_com_metricas(
    df_base,
    df_park,
    df_ae,
    y_ref_healthy,
    fcols,
    fhz,
    falha=0,
    temperatura=48,
    salvar=True
):
    aplicar_estilo_artigo()

    idx_show = selecionar_indice_exemplo(
        df_base=df_base,
        falha=falha,
        temperatura=temperatura
    )

    fhz_khz = fhz / 1e3

    T_real = df_base.loc[idx_show, "temperatura_c"]
    D_real = df_base.loc[idx_show, "falha"]

    y_original = df_base.loc[idx_show, fcols].to_numpy(float)
    y_park = df_park.loc[idx_show, fcols].to_numpy(float)
    y_ae = df_ae.loc[idx_show, fcols].to_numpy(float)

    print("\n================ CURVA EXEMPLO ================")
    print(f"Índice usado: {idx_show}")
    print(f"Falha: {D_real}")
    print(f"Temperatura: {T_real} °C")

    print("\n--- Original ---")
    print(f"RMSD: {rmsd(y_original, y_ref_healthy):.6f}")
    print(f"CCDM: {ccdm(y_original, y_ref_healthy):.6f}")

    print("\n--- Park ---")
    print(f"RMSD: {rmsd(y_park, y_ref_healthy):.6f}")
    print(f"CCDM: {ccdm(y_park, y_ref_healthy):.6f}")

    print("\n--- Autoencoder Curve Input ---")
    print(f"RMSD: {rmsd(y_ae, y_ref_healthy):.6f}")
    print(f"CCDM: {ccdm(y_ae, y_ref_healthy):.6f}")

    plt.figure(figsize=(12, 6), dpi=300)

    plt.plot(
        fhz_khz,
        y_ref_healthy,
        "--",
        c="black",
        linewidth=1.2,
        label=f"Referência saudável {REF_TEMP} °C"
    )

    plt.plot(
        fhz_khz,
        y_original,
        linewidth=1.0,
        alpha=0.65,
        label=f"Original — {T_real} °C — Dano {D_real}"
    )

    plt.plot(
        fhz_khz,
        y_park,
        linewidth=1.7,
        label="Park"
    )

    plt.plot(
        fhz_khz,
        y_ae,
        linewidth=1.9,
        label="Autoencoder Curve Input"
    )

    plt.xlabel("Frequência (kHz)")
    plt.ylabel("Parte real da impedância")

    plt.title(
        f"Exemplo de compensação — Dano {D_real} — {T_real} °C"
    )

    plt.legend(
        frameon=True,
        facecolor="white",
        edgecolor="none"
    )

    plt.grid(False)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    if salvar:
        caminho = os.path.join(
            OUTPUT_DIR,
            f"Curva_individual_Dano{D_real}_Temp{T_real}_idx{idx_show}.png"
        )

        plt.savefig(caminho, bbox_inches="tight", facecolor="white")

        print(f"Figura salva em: {caminho}")

    plt.show()


def plot_curvas_exemplo_grid(
    df_base,
    df_park,
    df_ae,
    y_ref_healthy,
    fcols,
    fhz,
    exemplos=((0, 48), (1, 55), (2, 55)),
    salvar=True
):
    aplicar_estilo_artigo()

    fhz_khz = fhz / 1e3

    fig, axes = plt.subplots(
        len(exemplos),
        1,
        figsize=(13, 5.0 * len(exemplos)),
        dpi=300,
        sharex=True
    )

    if len(exemplos) == 1:
        axes = [axes]

    for ax, (falha, temperatura) in zip(axes, exemplos):

        idx_show = selecionar_indice_exemplo(
            df_base=df_base,
            falha=falha,
            temperatura=temperatura
        )

        T_real = df_base.loc[idx_show, "temperatura_c"]
        D_real = df_base.loc[idx_show, "falha"]

        ax.plot(
            fhz_khz,
            y_ref_healthy,
            "--",
            c="black",
            linewidth=1.1,
            label=f"Referência saudável {REF_TEMP} °C"
        )

        ax.plot(
            fhz_khz,
            df_base.loc[idx_show, fcols],
            linewidth=1.0,
            alpha=0.65,
            label=f"Original — {T_real} °C"
        )

        ax.plot(
            fhz_khz,
            df_park.loc[idx_show, fcols],
            linewidth=1.7,
            label="Park"
        )

        ax.plot(
            fhz_khz,
            df_ae.loc[idx_show, fcols],
            linewidth=1.9,
            label="Autoencoder Curve Input"
        )

        ax.set_ylabel("Parte real da impedância")

        ax.set_title(
            f"Curva exemplo — Dano {D_real} — {T_real} °C"
        )

        ax.grid(False)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.legend(
            frameon=True,
            facecolor="white",
            edgecolor="none"
        )

    axes[-1].set_xlabel("Frequência (kHz)")

    plt.tight_layout()

    if salvar:
        caminho = os.path.join(
            OUTPUT_DIR,
            "Curvas_exemplo_grid.png"
        )

        plt.savefig(caminho, bbox_inches="tight", facecolor="white")

        print(f"Figura salva em: {caminho}")

    plt.show()


def plot_ae_loss(history, salvar=True):
    aplicar_estilo_artigo()

    plt.figure(figsize=(10, 5), dpi=300)

    plt.plot(
        history["epoch"],
        history["train_loss"],
        linewidth=2,
        label="Treino"
    )

    plt.plot(
        history["epoch"],
        history["val_loss"],
        linewidth=2,
        label="Validação"
    )

    plt.plot(
        history["epoch"],
        history["val_curve"],
        linewidth=2,
        label="Validação curva"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Treinamento do Autoencoder Curve Input")

    plt.legend(
        frameon=True,
        facecolor="white",
        edgecolor="none"
    )

    plt.grid(False)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    if salvar:
        caminho = os.path.join(
            OUTPUT_DIR,
            "Autoencoder_CurveInput_loss.png"
        )

        plt.savefig(caminho, bbox_inches="tight", facecolor="white")

        print(f"Figura salva em: {caminho}")

    plt.show()


def avaliar_auxiliares(df_comp):
    y_true = df_comp["falha"].to_numpy()
    y_pred = df_comp["falha_pred_aux"].to_numpy()

    print("\n==================== CLASSIFICAÇÃO AUXILIAR DE DANO ====================")
    print("Essa classificação é só auxiliar; a falha NÃO foi input da rede.")
    print("Matriz de confusão:")
    print(confusion_matrix(y_true, y_pred))
    print(f"ACC = {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1 macro = {f1_score(y_true, y_pred, average='macro'):.4f}")
    print("\nRelatório:")
    print(classification_report(y_true, y_pred, digits=4))

    if "temperatura_pred_aux" in df_comp.columns:
        temp_true = df_comp["temperatura_c"].to_numpy(float)
        temp_pred = df_comp["temperatura_pred_aux"].to_numpy(float)

        mae = np.mean(np.abs(temp_true - temp_pred))
        rmse = np.sqrt(np.mean((temp_true - temp_pred) ** 2))

        print("\n==================== PREDIÇÃO AUXILIAR DE TEMPERATURA ====================")
        print("A temperatura NÃO foi input da rede.")
        print(f"MAE temperatura = {mae:.4f} °C")
        print(f"RMSE temperatura = {rmse:.4f} °C")


# ============================================================
# 10) EXECUÇÃO PRINCIPAL
# ============================================================

def executar_comparacao_curve_input():
    timings = {}

    print("====================================================")
    print("PARK vs AUTOENCODER COM CURVA COMO ENTRADA")
    print("====================================================")

    t0 = time.time()

    df = pd.read_pickle(ARQ_BASE)

    fcols, fhz = get_freq_columns(
        df,
        FREQ_MIN_KHZ,
        FREQ_MAX_KHZ
    )

    ref_by_damage = get_reference_curves_by_damage(
        df=df,
        fcols=fcols,
        ref_temp=REF_TEMP
    )

    y_ref_healthy = ref_by_damage[0]

    timings["load_reference"] = time.time() - t0

    print(f"\nTotal de amostras: {len(df)}")
    print(f"Classes de dano: {sorted(df['falha'].unique())}")
    print(f"Amostras sem falha: {len(df[df['falha'] == 0])}")
    print(f"Faixa usada: {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
    print(f"Número de pontos de frequência: {len(fcols)}")
    print(f"Temperatura de referência: {REF_TEMP} °C")

    # Original
    t0 = time.time()

    df_original_metricas = calcular_metricas(
        df_curvas=df,
        fcols=fcols,
        y_ref_healthy=y_ref_healthy,
        metodo="Original"
    )

    timings["original_metrics"] = time.time() - t0

    # Park
    t0 = time.time()

    df_park_curvas = compensar_park(
        df=df,
        fcols=fcols,
        fHz=fhz,
        y_ref_healthy=y_ref_healthy
    )

    df_park_metricas = calcular_metricas(
        df_curvas=df_park_curvas,
        fcols=fcols,
        y_ref_healthy=y_ref_healthy,
        metodo="Park"
    )

    timings["park"] = time.time() - t0

    # Autoencoder
    t0 = time.time()

    df_ae_curvas, ae_extra = compensar_autoencoder_curva_input(
        df=df,
        fcols=fcols,
        ref_by_damage=ref_by_damage
    )

    df_ae_metricas = calcular_metricas(
        df_curvas=df_ae_curvas,
        fcols=fcols,
        y_ref_healthy=y_ref_healthy,
        metodo="Autoencoder"
    )

    timings["autoencoder_curve_input"] = time.time() - t0

    df_long = pd.concat(
        [
            df_original_metricas,
            df_park_metricas,
            df_ae_metricas
        ],
        axis=0,
        ignore_index=False
    )

    metodos_comp = ("Park", "Autoencoder")

    tabela_resumo = resumo_geral(
        df_long,
        metodos=metodos_comp
    )

    tabela_temp = resumo_por_temperatura(
        df_long,
        metodos=metodos_comp
    )

    df_mono, resumo_mono = checar_monotonicidade(
        df_long,
        metodos=metodos_comp
    )

    df_long.to_csv(
        os.path.join(OUTPUT_DIR, "df_long_metricas.csv"),
        index=False
    )

    tabela_temp.to_csv(
        os.path.join(OUTPUT_DIR, "resumo_por_temperatura.csv"),
        index=False
    )

    df_mono.to_csv(
        os.path.join(OUTPUT_DIR, "monotonicidade.csv"),
        index=False
    )

    resumo_mono.to_csv(
        os.path.join(OUTPUT_DIR, "resumo_monotonicidade.csv"),
        index=False
    )

    ae_extra["history"].to_csv(
        os.path.join(OUTPUT_DIR, "historico_loss_autoencoder_curve_input.csv"),
        index=False
    )

    df_ae_curvas[[
        "temperatura_c",
        "falha",
        "falha_pred_aux",
        "temperatura_pred_aux"
    ]].to_csv(
        os.path.join(OUTPUT_DIR, "predicoes_auxiliares.csv"),
        index=False
    )

    print("\n==================== RESUMO GERAL ====================")
    print(tabela_resumo)

    print("\n==================== MONOTONICIDADE ====================")
    print(resumo_mono)

    print("\n==================== TEMPOS ====================")
    for k, v in timings.items():
        print(f"{k:28s}: {v:.3f} s")

    avaliar_auxiliares(df_ae_curvas)

    print("\n✅ Comparação concluída.")

    return {
        "df_base": df,
        "df_park_curvas": df_park_curvas,
        "df_ae_curvas": df_ae_curvas,
        "df_original_metricas": df_original_metricas,
        "df_park_metricas": df_park_metricas,
        "df_ae_metricas": df_ae_metricas,
        "df_long": df_long,
        "tabela_resumo": tabela_resumo,
        "tabela_temp": tabela_temp,
        "df_mono": df_mono,
        "resumo_mono": resumo_mono,
        "y_ref_healthy": y_ref_healthy,
        "ref_by_damage": ref_by_damage,
        "fcols": fcols,
        "fhz": fhz,
        "ae_extra": ae_extra,
        "ae_history": ae_extra["history"],
        "timings": timings
    }


# ============================================================
# 11) RODAR TUDO
# ============================================================

resultados = executar_comparacao_curve_input()

df_base = resultados["df_base"]
df_park_curvas = resultados["df_park_curvas"]
df_ae_curvas = resultados["df_ae_curvas"]

df_original_metricas = resultados["df_original_metricas"]
df_park_metricas = resultados["df_park_metricas"]
df_ae_metricas = resultados["df_ae_metricas"]
df_long = resultados["df_long"]

tabela_resumo = resultados["tabela_resumo"]
tabela_temp = resultados["tabela_temp"]
df_mono = resultados["df_mono"]
resumo_mono = resultados["resumo_mono"]

y_ref_healthy = resultados["y_ref_healthy"]
ref_by_damage = resultados["ref_by_damage"]
fcols = resultados["fcols"]
fhz = resultados["fhz"]

ae_extra = resultados["ae_extra"]
ae_history = resultados["ae_history"]
timings = resultados["timings"]

metodos_comp = ("Park", "Autoencoder")


# ============================================================
# 12) GERAR GRÁFICOS
# ============================================================

plot_metricas_por_temperatura(
    df_long=df_long,
    dano=0,
    metodos=metodos_comp,
    salvar=True
)

plot_metricas_por_temperatura(
    df_long=df_long,
    dano=1,
    metodos=metodos_comp,
    salvar=True
)

plot_metricas_por_temperatura(
    df_long=df_long,
    dano=2,
    metodos=metodos_comp,
    salvar=True
)

plot_metricas_medias_por_dano(
    df_long=df_long,
    metodos=metodos_comp,
    salvar=True
)

plot_histogramas_metricas_grid(
    df_long=df_long,
    metodos=metodos_comp,
    bins=HIST_BINS,
    salvar=True
)

plot_ae_loss(
    history=ae_history,
    salvar=True
)

plot_curvas_exemplo_grid(
    df_base=df_base,
    df_park=df_park_curvas,
    df_ae=df_ae_curvas,
    y_ref_healthy=y_ref_healthy,
    fcols=fcols,
    fhz=fhz,
    exemplos=PLOT_EXEMPLOS,
    salvar=True
)

plot_curva_exemplo_individual_com_metricas(
    df_base=df_base,
    df_park=df_park_curvas,
    df_ae=df_ae_curvas,
    y_ref_healthy=y_ref_healthy,
    fcols=fcols,
    fhz=fhz,
    falha=0,
    temperatura=48,
    salvar=True
)

plot_curva_exemplo_individual_com_metricas(
    df_base=df_base,
    df_park=df_park_curvas,
    df_ae=df_ae_curvas,
    y_ref_healthy=y_ref_healthy,
    fcols=fcols,
    fhz=fhz,
    falha=1,
    temperatura=55,
    salvar=True
)

plot_curva_exemplo_individual_com_metricas(
    df_base=df_base,
    df_park=df_park_curvas,
    df_ae=df_ae_curvas,
    y_ref_healthy=y_ref_healthy,
    fcols=fcols,
    fhz=fhz,
    falha=2,
    temperatura=55,
    salvar=True
)


# ============================================================
# 13) MOSTRAR TABELAS
# ============================================================

print("\n==================== TABELA RESUMO ====================")
display(tabela_resumo)

print("\n==================== RESUMO POR TEMPERATURA ====================")
display(tabela_temp)

print("\n==================== MONOTONICIDADE POR TEMPERATURA ====================")
display(df_mono)

print("\n==================== RESUMO DA MONOTONICIDADE ====================")
display(resumo_mono)

print("\n✅ Todos os gráficos foram gerados e salvos.")